In [4]:

import pandas as pd
import numpy as np
import os
 
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
 
DATA_DIR = "."

In [7]:
#  1. Load & Inspect all 9 CSVs
files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
 
dfs = {}
for name, filename in files.items():
    path = os.path.join(DATA_DIR, filename)
    dfs[name] = pd.read_csv(path)
    print(f"--- {name} ---")
    print("shape:", dfs[name].shape)
    print(dfs[name].dtypes)
    print(dfs[name].isnull().sum()[dfs[name].isnull().sum() > 0])
    print()
 
orders = dfs["orders"]
order_items = dfs["order_items"]
order_payments = dfs["order_payments"]
order_reviews = dfs["order_reviews"]
customers = dfs["customers"]
sellers = dfs["sellers"]
products = dfs["products"]
geolocation = dfs["geolocation"]
category_translation = dfs["category_translation"]
 

--- orders ---
shape: (99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- order_items ---
shape: (112650, 7)
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object
Series([], dtype: int64)

--- order_payments ---
shape: (103886, 5)
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: 

In [8]:

# 2. Datetime conversion
 
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
 
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")
 
orders[date_cols].dtypes
 

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [9]:

# 3. Missing value handling

orders["delivery_status"] = np.where(
    orders["order_delivered_customer_date"].isna(),
    "Not Delivered",
    "Delivered"
)
 
print(orders["delivery_status"].value_counts())
print(orders["order_status"].value_counts())

delivery_status
Delivered        96476
Not Delivered     2965
Name: count, dtype: int64
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [10]:

# 4. Feature engineering

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days
 
orders["is_late"] = np.where(
    orders["delivery_status"] == "Delivered",
    orders["delivery_delay_days"] > 0,
    np.nan  
)
 
orders["processing_time_days"] = (
    orders["order_delivered_carrier_date"] - orders["order_purchase_timestamp"]
).dt.days
 
orders[["order_id", "delivery_status", "delivery_delay_days", "is_late", "processing_time_days"]].head()
 

,order_id,delivery_status,delivery_delay_days,is_late,processing_time_days
0,e481f51cbdc54678b7cc49136f2d6af7,Delivered,-8.0,0.0,2.0
1,53cdb2fc8bc7dce0b6741e2150273451,Delivered,-6.0,0.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,Delivered,-18.0,0.0,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,Delivered,-13.0,0.0,3.0
4,ad21c59c0840e6cb83a9ceb5573f8159,Delivered,-10.0,0.0,0.0


In [11]:

# 5. Deduplication check

dup_check = order_items.groupby("order_id").size().reset_index(name="n_items")
print("Orders with more than 1 item:", (dup_check["n_items"] > 1).sum())
print(dup_check["n_items"].describe())
 
order_items_agg = order_items.groupby("order_id").agg(
    total_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
).reset_index()
 
true_dupes = order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
print("True duplicate rows:", true_dupes)
 

Orders with more than 1 item: 9803
count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: n_items, dtype: float64
True duplicate rows: 0


In [12]:

# 6. Text cleaning — normalize product category names

products["product_category_name"] = (
    products["product_category_name"].str.strip().str.lower()
)
 
category_translation["product_category_name"] = (
    category_translation["product_category_name"].str.strip().str.lower()
)
category_translation["product_category_name_english"] = (
    category_translation["product_category_name_english"].str.strip().str.lower()
)
 
products = products.merge(category_translation, on="product_category_name", how="left")
 
missing_translation = products["product_category_name_english"].isna().sum()
print("Products with untranslated category:", missing_translation)
products["product_category_name_english"] = products["product_category_name_english"].fillna("unknown")

Products with untranslated category: 623


In [13]:

# 7. Outlier check — freight value & price (IQR method)

def flag_outliers_iqr(df, col, factor=1.5):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return (df[col] < lower) | (df[col] > upper)
 
order_items["price_outlier"] = flag_outliers_iqr(order_items, "price")
order_items["freight_outlier"] = flag_outliers_iqr(order_items, "freight_value")
 
print("Price outliers:", order_items["price_outlier"].sum())
print("Freight outliers:", order_items["freight_outlier"].sum())

Price outliers: 8427
Freight outliers: 12134


In [14]:

#  8. Merge into analysis-ready tables

orders_fact = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(order_items_agg, on="order_id", how="left")
)
 
# Item-level fact table (for seller/product-grain analysis)
items_fact = (
    order_items
    .merge(products[["product_id", "product_category_name_english"]], on="product_id", how="left")
    .merge(sellers, on="seller_id", how="left")
    .merge(orders[["order_id", "customer_id", "order_purchase_timestamp",
                    "delivery_status", "is_late"]], on="order_id", how="left")
)
 
# Reviews, deduplicated to one row per order
reviews_clean = (
    order_reviews
    .sort_values("review_creation_date")
    .drop_duplicates(subset="order_id", keep="last")
)
 
print("orders_fact:", orders_fact.shape)
print("items_fact:", items_fact.shape)
print("reviews_clean:", reviews_clean.shape)
 

orders_fact: (99441, 19)
items_fact: (112650, 17)
reviews_clean: (98673, 7)


In [15]:

# 9. Export final cleaned tables to CSV

os.makedirs("clean_data", exist_ok=True)
 
orders_fact.to_csv("clean_data/orders_fact.csv", index=False)
items_fact.to_csv("clean_data/items_fact.csv", index=False)
reviews_clean.to_csv("clean_data/reviews_clean.csv", index=False)
customers.to_csv("clean_data/customers_clean.csv", index=False)
sellers.to_csv("clean_data/sellers_clean.csv", index=False)
products.to_csv("clean_data/products_clean.csv", index=False)
 
print("Clean CSV files written to ./clean_data/")

Clean CSV files written to ./clean_data/
